# Notebook 02 — Pattern Matching (`LIKE`, `REGEXP`), Aggregation (`GROUP BY`, `HAVING`), and Window Functions (`OVER`)

**Session:** Database - SQL (1)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Practice string pattern matching with `LIKE` wildcards (`%`, `_`) and regular expressions (`REGEXP`), summarize groups with aggregate functions and `GROUP BY` / `HAVING`, and contrast group-collapsing aggregation against row-preserving window functions (`OVER (PARTITION BY ... ORDER BY ...)`).

In [ ]:
import re
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))

# Register SQLite REGEXP function backed by Python's re.search (case-insensitive to match MySQL default)
conn.create_function(
    "REGEXP",
    2,
    lambda pattern, value: (
        bool(re.search(pattern, value, re.IGNORECASE)) if value is not None else False
    ),
)


def query(sql: str) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))

## 1. Wildcard Pattern Matching (`LIKE`)

In [ ]:
# Last name starts with 'B'
show_rows(query("SELECT customer_id, last_name FROM customers WHERE last_name LIKE 'B%';"))

# Last name is exactly 6 characters and ends with 'y'
show_rows(query("SELECT customer_id, last_name FROM customers WHERE last_name LIKE '_____y';"))

# Address contains 'TRAIL' or 'AVENUE'
show_rows(
    query(
        """
        SELECT customer_id, address
        FROM customers
        WHERE address LIKE '%trail%' OR address LIKE '%avenue%';
        """
    )
)

# Phone has '9' four characters from the end
show_rows(query("SELECT customer_id, phone FROM customers WHERE phone LIKE '%9___';"))

## 2. Regular Expressions (`REGEXP`)

In [ ]:
# First name is Elka or Ambur
show_rows(
    query("SELECT customer_id, first_name FROM customers WHERE first_name REGEXP 'ELKA|AMBUR';")
)

# Last name ends with EY or ON
show_rows(query("SELECT customer_id, last_name FROM customers WHERE last_name REGEXP 'EY$|ON$';"))

# Last name starts with MY or contains SE
show_rows(query("SELECT customer_id, last_name FROM customers WHERE last_name REGEXP '^MY|SE';"))

# Last name contains B followed by R or U
show_rows(query("SELECT customer_id, last_name FROM customers WHERE last_name REGEXP 'B[RU]';"))

## 3. Aggregate Functions, `GROUP BY`, and `HAVING`

In [ ]:
# Whole-table summaries
show_rows(query("SELECT ROUND(AVG(points), 2) AS average_point FROM customers;"))
show_rows(query("SELECT SUM(quantity_in_stock) AS total_quantity FROM products;"))

# Row-collapsing GROUP BY + post-aggregation HAVING
state_summary = query(
    """
    SELECT
        state,
        COUNT(*) AS customer_count,
        ROUND(AVG(points), 2) AS average_state_point,
        SUM(points) AS total_state_point
    FROM customers
    GROUP BY state
    HAVING total_state_point > 3000
    ORDER BY total_state_point DESC;
    """
)
show_rows(state_summary)

## 4. Window Functions (`OVER (PARTITION BY ... ORDER BY ...)`)

In [ ]:
# Row-preserving state average + cumulative state sum
window_rows = query(
    """
    SELECT
        customer_id,
        state,
        points,
        ROUND(AVG(points) OVER (PARTITION BY state), 2) AS avg_state_point,
        SUM(points) OVER (PARTITION BY state ORDER BY points) AS cumulative_state_points
    FROM customers
    ORDER BY state, points;
    """
)
show_rows(window_rows)